In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# 1. Načtení dat
df = pd.read_csv('instagram data.csv')
print(f"Původní rozměr dat: {df.shape}")
display(df.head(3))

Původní rozměr dat: (100000, 26)


,reel_id,creator_id,creator_followers,posting_time,reel_length_sec,hook_strength_score,caption_length,hashtags_count,trending_audio,audio_popularity_score,...,likes,comments,shares,saves,engagement_rate,impressions,reach,non_follower_reach_ratio,explore_page_boost,virality_score
0,c437715e-d1e4-4f40-8548-8718243c6516,70c2ca5a-9d3a-4ba0-9aed-62cdbbc1663f,4208388,3,31,0.451657,0,20,True,0.399659,...,779301,30002,16741,21305,0.884296,75968410,39007279,0.960121,False,82.843160
1,9097bb11-0e03-4360-a239-6613ed2025df,b6bb4240-6341-47ce-bdd4-86fcba18d9ad,4096880,4,38,0.988437,205,2,False,0.294848,...,482481,49717,10095,19802,0.609125,90722120,84132612,0.909716,False,60.256519
2,a8e09994-3314-400b-9f3a-683b494f1070,7902b14d-cb95-4673-8ef1-482ff5151801,2624351,0,13,0.019949,205,6,True,0.697575,...,925249,12046,13883,13444,0.057583,75215994,82019789,0.101694,True,0.483295


In [2]:
# 2. Příprava cílového atributu a odstranění identifikátorů
# Uložíme si ID pro pozdější dohledání konkrétní instance, ale z prediktorů je vyřadíme
# (unikátní UUID by způsobila přeučení stromů bez jakékoli obecné platnosti)
ids = df[['reel_id', 'creator_id']].copy()

# Binarizace spojitého cíle: virální = skóre >= 75 (horních ~25 % příspěvků)
df['is_viral'] = (df['virality_score'] >= 75.0).astype(int)

# Odstranění původního cílového atributu a ID sloupců
df_clean = df.drop(columns=['virality_score', 'reel_id', 'creator_id'])

# Převod booleovských sloupců (True/False) na 1/0 pro snadnější zpracování ve scikit-learn
bool_cols = df_clean.select_dtypes(include=['bool']).columns
df_clean[bool_cols] = df_clean[bool_cols].astype(int)

print("Rozložení cílové proměnné is_viral:")
print(df_clean['is_viral'].value_counts(normalize=True).round(4))

Rozložení cílové proměnné is_viral:
is_viral
0    0.7471
1    0.2529
Name: proportion, dtype: float64


In [3]:
# 3. Rozdělení na trénovací a testovací množinu (80:20)
X = df_clean.drop(columns=['is_viral'])
y = df_clean['is_viral']

# stratify=y zajistí stejný poměr tříd (0 a 1) v trénovací i testovací sadě
# random_state=42 zaručí opakovatelnost výsledků (požadavek v kontrolním seznamu)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f"Trénovací sada: {X_train.shape[0]} řádků")
print(f"Testovací sada: {X_test.shape[0]} řádků")

Trénovací sada: 80000 řádků
Testovací sada: 20000 řádků


In [4]:
# 4. Škálování (standardizace) bez úniku dat (data leakage)
# Poznámka: Rozhodovací stromy škálování nutně nepotřebují, ale v zadání je doporučeno
# a zároveň ho využijete pro porovnání vlivu předzpracování v sekci Evaluace.
scaler = StandardScaler()

# DŮLEŽITÉ: Fitujeme POUZE na trénovacích datech!
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Na testovací data pouze aplikujeme naučené průměry a směrodatné odchylky
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

# Výběr jedné konkrétní instance z testovací sady pro pozdější analýzu
vybrany_index = X_test.index[0]
print(f"Vybraná instance má index {vybrany_index} a reel_id: {ids.loc[vybrany_index, 'reel_id']}")

Vybraná instance má index 63769 a reel_id: 90464090-f30f-4351-a667-e92d07f6b2bb


In [5]:
# 5. Předzpracování podmnožiny dat pro shlukování (modely bez učitele)
# Vybereme podmnožinu: příspěvky s trendujícím audiem (trending_audio == 1)
df_cluster_raw = df_clean[df_clean['trending_audio'] == 1].copy()

# Pro shlukování odstraníme cílový atribut (a konstantní sloupec trending_audio)
X_cluster = df_cluster_raw.drop(columns=['is_viral', 'trending_audio'])

# Pro shlukování (zejména k-means a hierarchické) je standardizace nutná kvůli různým měřítkům
# (např. impressions v desítkách milionů vs. retention_rate v intervalu 0-1)
cluster_scaler = StandardScaler()
X_cluster_scaled = pd.DataFrame(
    cluster_scaler.fit_transform(X_cluster),
    columns=X_cluster.columns,
    index=X_cluster.index
)

print(f"Velikost podmnožiny pro shlukování: {X_cluster_scaled.shape}")

Velikost podmnožiny pro shlukování: (49917, 22)


In [6]:
# 6. Uložení 3 předzpracovaných souborů pro odevzdání
train_export = X_train_scaled.copy()
train_export['is_viral'] = y_train
train_export.to_csv('data_train_processed.csv', index=True)

test_export = X_test_scaled.copy()
test_export['is_viral'] = y_test
test_export.to_csv('data_test_processed.csv', index=True)

X_cluster_scaled.to_csv('data_clustering_processed.csv', index=True)
print("Všechny 3 předzpracované datové sady byly úspěšně uloženy.")

Všechny 3 předzpracované datové sady byly úspěšně uloženy.
